# White Castle rules assistant

An assistant that answers questions about the rules of the board game **White Castle**, in Spanish. It uses RAG (retrieval-augmented generation): instead of answering from memory, it first looks up the relevant passages of the official rulebook and then answers from them, citing the page.

**The starting point** is the Spanish rulebook: a 20-page scanned PDF where every page is an image, with no text to search. The notebook follows the path that turns it into an assistant:

| Section | Step | What happens |
|---|---|---|
| 1.1 | PDF → text | Gemini transcribes each page image to Markdown |
| 1.2 | Text → chunks | The text is cleaned and split into chunks that keep their page and section |
| 1.3 | Chunks → vectors | Gemini embeddings, stored in a local ChromaDB collection |
| 1.4 | Question → rules | Test questions check that the search finds the right rules |
| 2 | Rules → answer | LangGraph agent: retrieve the rules, then generate the answer with Gemini |
| 3 | Answer → conversation | Memory between turns, for follow-up questions |
| 4 | Conversation | Chat cell and example questions |

How to install and run it (dependencies, Gemini API key, where to place the rulebook): see the README.

## 0. Setup

Imports, the configuration shared by the whole notebook (paths and models) and the Gemini API key, read from `.env` and never printed.

The parameters of each step live in its own section, next to the explanation of their values.

In [1]:
# Imports
import base64
import os
import re
import statistics
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pypdf import PdfReader

In [2]:
# Paths
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
PDF_PATH = PROJECT_DIR / "data" / "white_castle.pdf"
TRANSCRIPT_DIR = PROJECT_DIR / "data" / "transcript"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

# Models
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"
CHAT_REQUESTS_PER_SECOND = 0.2  # 12 per minute: the free tier allows 15

In [3]:
def load_api_key():
    """Gemini API key: .env → environment variable GOOGLE_API_KEY."""
    load_dotenv(ENV_PATH)
    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError(f"GOOGLE_API_KEY is empty or missing in {ENV_PATH}")


load_api_key()

## 1. Knowledge base

The agent can only answer as well as it searches. This part turns the scanned rulebook into a knowledge base the agent can search, and ends by checking that the search finds the right rules.

### 1.1 From a scanned PDF to text

The PDF has no text layer: each page is a single JPEG image. Instead of OCR, Gemini reads each image and transcribes it to Markdown:
- **One page per request:** with the whole PDF at once the model tends to summarise, and the page of each rule would be lost. If a page fails, only that page is repeated.
- **Icons that carry rules are described:** in a board game many rules (costs, rewards) only appear as icons, so they are described briefly between brackets, for example `[Icono: 2 monedas]`. Decorative illustrations are ignored.
- **Done once:** each page is saved to `data/transcript/page_NN.md` and skipped in later runs, so the quota is only spent once.

In [4]:
# Constants
TRANSCRIPTION_PROMPT = """You are transcribing one page of the Spanish rulebook of the board game White Castle.
- Transcribe all the text literally, in Spanish. Do not translate, summarize or correct it.
- Mark section titles with Markdown headings: # for main titles, ## for subsections.
- When an icon, diagram or illustrated example carries rules information, describe it briefly in Spanish between square brackets, for example: [Icono: 2 monedas].
- Ignore purely decorative illustrations, page numbers and repeated headers or footers.
- Return only the transcription, with no comments before or after it."""

In [5]:
def transcribe_manual():
    """Rulebook PDF → one Markdown file per page in TRANSCRIPT_DIR (pages already done are skipped)."""
    TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    pages = PdfReader(PDF_PATH).pages
    transcribed = [transcribe_page(llm, page, number) for number, page in enumerate(pages, start=1)]
    print(f"{len(pages)} pages: {sum(transcribed)} transcribed now, {len(pages) - sum(transcribed)} already on disk")


def transcribe_page(llm, page, number):
    """One page: scanned image → Gemini transcription → page_NN.md (False if it was already on disk)."""
    output_path = TRANSCRIPT_DIR / f"page_{number:02d}.md"
    if output_path.exists():
        return False
    message = build_page_message(page)
    reply = llm.invoke([message])
    output_path.write_text(reply.text, encoding="utf-8")
    print(f"Page {number:02d}: transcribed ({len(reply.text)} characters)")
    return True


def build_page_message(page):
    """Message for Gemini: transcription prompt + the page JPEG image as base64."""
    image_b64 = base64.b64encode(page.images[0].data).decode()
    return HumanMessage(content=[
        {"type": "text", "text": TRANSCRIPTION_PROMPT},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
    ])


transcribe_manual()

20 pages: 0 transcribed now, 20 already on disk


### 1.2 Cleaning and chunking

The agent does not search whole pages but chunks: pieces small enough to hold one rule each. The transcription becomes chunks in four steps:
- **Cleaning:** removes the transcription noise that repeats (formula marks, arrow bullets, warning signs, decorative images). Isolated typos are left as they are: fixing them by hand does not pay off for the search.
- **Sections first:** the text is split by its Markdown headings, so a rule is not cut in two. A page that starts mid-section keeps the section of the previous page.
- **Then size:** sections longer than `CHUNK_SIZE` characters are split again, with some overlap so that a sentence on the border is not lost.
- **Each chunk carries its context:** its page and section are saved as metadata, to cite the source. The section path is also written at the start of the text, because the embedding only sees the text: a loose sentence like "pay 2 coins" means little without its section.

In [6]:
# Constants
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

# Cleaning rules: (regex pattern, replacement)
CLEANING_RULES = [
    (r"\$([^$\n]+)\$",                        r"\1"),  # formula marks: $10+10$ → 10+10
    (r"\[Icono: Flecha roja\]\s*",            "- "),   # arrow bullets → list items
    (r"\[Icono: Signo de exclamación\]\s*",   ""),     # warning sign before "Atención"
    (r"\*?\[Ilustración decorativa:[^\]]*\]\*?", ""),  # decorative images
    (r"\n{3,}",                               "\n\n"), # extra blank lines
]

In [7]:
def build_chunks():
    """Transcript pages → cleaned text → sections → chunks with page and section metadata."""
    pages = load_pages()
    sections = split_into_sections(pages)
    chunks = split_long_sections(sections)
    print_chunk_summary(chunks)
    return chunks


def load_pages():
    """Transcript files → list of (page number, cleaned text)."""
    pages = []
    for path in sorted(TRANSCRIPT_DIR.glob("page_*.md")):
        number = int(path.stem.split("_")[1])
        pages.append((number, clean_text(path.read_text(encoding="utf-8"))))
    return pages


def clean_text(text):
    """Raw transcription → text without the noise listed in CLEANING_RULES."""
    for pattern, replacement in CLEANING_RULES:
        text = re.sub(pattern, replacement, text)
    return text.strip()


def split_into_sections(pages):
    """
    Pages → one document per section:
        - Metadata: headings (h1, h2, h3) and page number
        - Text at the top of a page with no heading keeps the section of the previous page
    """
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS)
    sections = []
    last_headers = {}
    for number, text in pages:
        for section in splitter.split_text(text):
            if section.metadata:
                last_headers = dict(section.metadata)
            else:
                section.metadata = dict(last_headers)
            section.metadata["section"] = " > ".join(section.metadata.values())
            section.metadata["page"] = number
            sections.append(section)
    return sections


def split_long_sections(sections):
    """Sections → chunks of at most CHUNK_SIZE characters, each starting with its section title."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(sections)
    for chunk in chunks:
        chunk.page_content = f"{chunk.metadata['section']}\n\n{chunk.page_content}"
    return chunks


def print_chunk_summary(chunks):
    """Chunk count, length statistics and one example chunk."""
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"Chunks: {len(chunks)}")
    print(f"Length (characters): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
    print(f"\nExample (chunk 10, page {chunks[10].metadata['page']}):\n{chunks[10].page_content}")


chunks = build_chunks()

Chunks: 76
Length (characters): min 118, median 616, max 1087

Example (chunk 10, page 2):
PREPARACIÓN DE LA PARTIDA > PUENTES

7 [Icono: 7 en cuadro rojo] Se colocan los puentes en sus espacios del río (si lo preferís, podéis jugar sin los puentes troquelados). Tomad 3/4/5 dados de cada color en función de si seis 2/3/4 jugadores. Devolved a la caja los dados que sobren.  
Tirad los dados de cada color y colocadlos en el puente que les corresponda, ordenándolos de manera ascendente según su valor de izquierda a derecha. Es decir, el valor más bajo a la izquierda del puente junto a la linterna, y el más alto a la derecha. Los demás se colocan en el centro del puente, en orden.


### 1.3 Vector store

Each chunk is turned into a vector with Gemini embeddings: chunks that talk about similar things get close vectors, so a question can find its rules by meaning and not only by exact words. The vectors are stored in a local ChromaDB collection (`chroma_db/`):
- If the collection already exists, it is loaded from disk and no embeddings are spent
- After changing the cleaning or chunking settings, set `REBUILD_VECTOR_STORE = True` once to rebuild it: the collection cannot tell by itself that the chunks have changed

In [8]:
# Constants
COLLECTION_NAME = "white_castle_rules"
REBUILD_VECTOR_STORE = False

In [9]:
def load_vector_store(chunks):
    """Chunks → ChromaDB collection in CHROMA_DIR (loaded from disk if it is already built)."""
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL),
        persist_directory=str(CHROMA_DIR),
    )
    stored = len(vector_store.get()["ids"])
    if REBUILD_VECTOR_STORE or stored == 0:
        index_chunks(vector_store, chunks)
    else:
        print(f"Vector store loaded from disk: {stored} chunks")
    return vector_store


def index_chunks(vector_store, chunks):
    """Empty the collection, then embed and store every chunk with a stable id."""
    vector_store.reset_collection()
    ids = [f"chunk_{number:03d}" for number in range(len(chunks))]
    vector_store.add_documents(chunks, ids=ids)
    print(f"Vector store built: {len(chunks)} chunks indexed")


vector_store = load_vector_store(chunks)

Vector store loaded from disk: 76 chunks


### 1.4 Does the search find the right rules?

Before building the agent on top of it, the search is checked with questions whose answer is known to be on a given page of the rulebook: direct questions about key rules, and harder ones written as a player would ask them, which need several rules at once.

For each question, the table shows the expected pages, the positions among the top `RETRIEVAL_K` chunks where a chunk from those pages appears, and the top chunk.

In [10]:
# Constants
RETRIEVAL_K = 5

LONG_DICE_QUESTION = (
    "En The White Castle, un jugador quiere coger un dado de valor 2 y colocarlo en un espacio cuyo valor de referencia es 5. "
    "Tiene 1 moneda, por lo que no puede pagar las 3 monedas de diferencia. Sin embargo, al coger ese dado "
    "activa su bonificación de linterna, que le proporcionaría monedas suficientes para cubrir la diferencia. "
    "¿Puede realizar la jugada? Explica el orden exacto de resolución, si puede utilizar la recompensa de la "
    "linterna para pagar el dado y qué alternativa tiene si no puede efectuar esa colocación."
)

LONG_GARDENER_QUESTION = (
    "Al final de la primera ronda, un jugador activa un jardinero que ya tenía colocado. La acción de ese jardinero "
    "le permite colocar un segundo jardinero en otro jardín bajo un puente que contiene dados. El jugador sostiene que, "
    "como el segundo jardinero también está bajo un puente con dados, debe activar inmediatamente su acción de final de "
    "ronda. Además, afirma que no debe pagar de nuevo el coste de arroz de ese segundo jardinero, pero que tampoco "
    "necesita pagar las monedas exigidas por la acción que decida ejecutar. Evalúa por separado las tres afirmaciones. "
    "Después, explica qué sucede si la misma situación se produce al final de la tercera ronda."
)

# Direct questions: (question, rulebook pages with the answer)
DIRECT_QUESTIONS = [
    ("¿Se pueden apilar dados en una partida a 2 jugadores?",              (5,)),
    ("¿Qué pasa si coloco un dado de valor inferior al de la casilla?",     (5,)),
    ("¿Cuántos cortesanos caben en la puerta del castillo?",                (10,)),
    ("¿Cuál es el máximo de recursos de cada tipo que puedo tener?",        (5,)),
    ("¿Cuánto cuesta subir un cortesano dos pisos?",                        (10,)),
    ("¿Cómo puntúan los guerreros al final de la partida?",                 (14,)),
    ("En el modo solitario, ¿el clan rival puntúa los recursos sobrantes?", (18,)),
]

# Harder questions, as a player would ask them: (question, rulebook pages with the answer)
HARD_QUESTIONS = [
    ("¿Puedes pagar un dado con los recursos que obtienes de la propia acción?",            (5, 8)),
    ("¿Puedes ascender dos veces o colocar dos cortesanos en una misma acción?",            (10,)),
    ("¿Puede un jardinero recién colocado activarse inmediatamente al final de una ronda?", (13, 10)),
    (LONG_DICE_QUESTION,                                                                    (5, 8)),   # full game situation, "linterna" for "lámpara"
    (LONG_GARDENER_QUESTION,                                                                (13, 10)), # full game situation, three claims to check
]

TEST_QUESTIONS = DIRECT_QUESTIONS + HARD_QUESTIONS

In [11]:
def check_retrieval():
    """Every test question → one table row, then how many questions found an expected page."""
    print(f"{'Question':<58} {'Expected':<9} {'Found at':<10} Top chunk")
    found = 0
    for question, expected_pages in TEST_QUESTIONS:
        chunks = vector_store.similarity_search(question, k=RETRIEVAL_K)
        ranks = [rank for rank, chunk in enumerate(chunks, start=1) if chunk.metadata["page"] in expected_pages]
        found += bool(ranks)
        print_row(question, expected_pages, ranks, chunks[0])
    print(f"\n{found}/{len(TEST_QUESTIONS)} questions with a chunk from an expected page in the top {RETRIEVAL_K}")


def print_row(question, expected_pages, ranks, top_chunk):
    """
    One aligned table row:
        Example: '¿Cuánto cuesta subir...'  p10  1, 3, 4  p10 CORTESANOS
        - Questions longer than the column are shortened
        - Icon descriptions are removed from the section title
    """
    short_question = question if len(question) <= 56 else question[:53] + "..."
    expected = ", ".join(f"p{page}" for page in expected_pages)
    found_at = ", ".join(str(rank) for rank in ranks) or "-"
    section = re.sub(r"\s*\[Icono:[^\]]*\]", "", top_chunk.metadata["section"])
    top = f"p{top_chunk.metadata['page']} {section[:45]}"
    print(f"{short_question:<58} {expected:<9} {found_at:<10} {top}")


check_retrieval()

Question                                                   Expected  Found at   Top chunk


¿Se pueden apilar dados en una partida a 2 jugadores?      p5        1, 3       p5 APILAR DADOS:


¿Qué pasa si coloco un dado de valor inferior al de l...   p5        1, 3       p5 USO DE LOS DADOS > PROCEDIMIENTO PARA USAR LO


¿Cuántos cortesanos caben en la puerta del castillo?       p10       1, 2, 3, 4 p10 CORTESANOS > A


¿Cuál es el máximo de recursos de cada tipo que puedo...   p5        1          p5 OBTENER Y PAGAR RECURSOS:


...

¿Puedes pagar un dado con los recursos que obtienes d...   p5, p8    1, 2, 3, 4 p8 EL DOMINIO PERSONAL


¿Puedes ascender dos veces o colocar dos cortesanos e...   p10       1, 2, 3, 4 p10 CORTESANOS


¿Puede un jardinero recién colocado activarse inmedia...   p13, p10  1, 2, 3, 5 p13 FINAL DE LA RONDA


En The White Castle, un jugador quiere coger un dado ...   p5, p8    3, 5       p6 EL CASTILLO


Al final de la primera ronda, un jugador activa un ja...   p13, p10  1, 2, 3, 5 p13 FINAL DE LA RONDA

12/12 questions with a chunk from an expected page in the top 5


**What the check shows**
- **Direct questions:** the top chunk comes from the expected page in all seven.
- **Harder questions, short form:** the top chunk also comes from an expected page. When the answer is spread over two pages (paying for a die: pages 5 and 8; gardeners: pages 13 and 10), both pages appear in the top 5.
- **Long questions:** the gardener one finds both pages in the first two positions. The dice one is the weakest case: the top chunk is *El castillo* (page 6), pulled by the words "The White Castle" of the question, and the expected chunks appear in positions 3 (paying the difference) and 5 (the order of the steps).
- **Why `RETRIEVAL_K = 5`:** with 3 chunks, the long dice question would lose the order of the steps, which is half of its answer. Five chunks of at most about 1,100 characters are still a small context for Gemini.
- **The search brings the pieces, not the answer:** in the harder questions the rulebook never states the answer literally (for example, "the lamp reward cannot pay for the die"). It has to be deduced from the order of the steps, which is the job of the agent and its system prompt (section 2).

## 2. RAG agent

The agent is a LangGraph graph: a small pipeline where a shared **state** travels through **nodes** (plain Python functions) following the **edges** between them. Each node reads the state and adds its result to it.

```
START → retrieve → generate → END
```

- **retrieve:** the question → the top `RETRIEVAL_K` chunks of the rulebook (section 1)
- **generate:** system prompt + chunks + question → Gemini's answer

For two steps in a row a graph is not strictly needed, but it is the base for section 3: the graph itself saves the memory of each conversation, and new steps (such as rewriting a follow-up question before searching) are just new nodes.

### 2.1 System prompt

The system prompt sets the role of the agent, the format of its answers and its limits. Each rule has a reason:

| Rule | Why |
|---|---|
| Expert, short answers | Players ask in the middle of a game: they need the answer, not an essay |
| Spanish answers, English prompt | The rulebook and the players are Spanish; models follow instructions in English best |
| Only the excerpts | Without it, the model fills the gaps with what it believes about the game, which may be wrong |
| Kind of question first | Each kind of question has its own format. With all the rules in one list, the model added a page even to questions it could not answer |
| Short answer → explanation → page | The first sentence settles the doubt at the table, the paragraph explains why, and the page lets the player check it |
| Limits read literally | The model misread the limit of the courtier action (see 2.3) |
| Deduce | Tricky answers are rarely written literally in the rulebook (see 1.4) |
| "No encuentro esta regla en el manual" + house rule | With only a few chunks, the agent cannot tell a gap in the rulebook from a failed search: the sentence is honest about both. The house rule keeps the game going, labelled as unofficial |
| One step at a time to learn the game | A whole explanation of the game is a wall of text. It needs the memory of section 3 |
| Fixed reply outside White Castle | Keeps the agent in its role. Worded freely, the model still added a page |

In [12]:
# Constants
SYSTEM_PROMPT = """You are an expert on the rules of the board game White Castle. You help players during a game, so your answers are short and clear. Always answer in Spanish.

You answer only from the rulebook excerpts given with each question. Each excerpt shows its page and section. Never invent a rule or answer from general knowledge of the game.

First decide which of these four kinds of question it is, and answer only as its section says.

# 1. Rules question answered by the excerpts
Answer in three parts:
1. Short answer, in the first sentence.
   - Yes/no questions: start with "Sí" or "No".
   - Any other question: answer it in one sentence.
2. Explanation: one short paragraph with the details and the reasoning, so the player understands why.
3. Source: the pages of the excerpts you used, for example "Lo tienes en la página 5."

Read quantities and limits literally (once, each, at most, only one): they are often the key to the answer. If the answer is not written literally but follows from the excerpts (for example, from the order of the steps), deduce it and explain the reasoning in the paragraph.

# 2. Rules question the excerpts do not answer
1. Start with "No encuentro esta regla en el manual" and say briefly that the rulebook may leave it open, or the search may not have found it. Never present an assumption as an official rule.
2. Propose a house rule for this game, clearly labelled as a suggestion, not an official rule.
Do not add a "Lo tienes en la página..." line: no page has the rule.

# 3. Learning to play
For broad questions, such as "enséñame a jugar", "cómo se prepara la partida" or "cómo se juega":
1. List the main steps as a short numbered stepper, one line per step.
2. Ask if the player is ready for the first step.
3. Explain one step at a time, briefly, with its page. Move to the next step only when the player confirms.
Never explain all the steps at once.

# 4. Not about White Castle
Reply only with "Solo puedo ayudarte con las reglas de White Castle.", with no explanation and no page."""

### 2.2 The graph

The state holds the conversation (`messages`) and the chunks retrieved for the last question. Each chunk reaches Gemini under a header with its page (`[Página 5]`), followed by its section and its text, so the answer can cite the page.

The chat model spaces its calls (`CHAT_REQUESTS_PER_SECOND`): the free tier allows 15 per minute, and running the whole notebook makes more than that.

In [13]:
class AgentState(TypedDict):
    """
    State that travels through the graph:
        - messages: the conversation, user questions and agent answers
        - chunks: the rulebook chunks retrieved for the last question
    """
    messages: Annotated[list, add_messages]
    chunks: list


def build_agent():
    """Graph START → retrieve → generate → END, compiled and ready to invoke."""
    graph = StateGraph(AgentState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)
    return graph.compile()


def retrieve(state):
    """Retrieve node: last user question → top RETRIEVAL_K chunks of the rulebook."""
    question = state["messages"][-1].content
    return {"chunks": vector_store.similarity_search(question, k=RETRIEVAL_K)}


def generate(state):
    """Generate node: system prompt + earlier messages + question with its chunks → Gemini answer."""
    *history, question = state["messages"]
    prompt = [SystemMessage(SYSTEM_PROMPT), *history, HumanMessage(build_user_message(question.content, state["chunks"]))]
    return {"messages": [llm.invoke(prompt)]}


def build_user_message(question, chunks):
    """Question + its chunks, each one under a header with its page."""
    excerpts = "\n\n".join(f"[Página {chunk.metadata['page']}]\n{chunk.page_content}" for chunk in chunks)
    return f"Rulebook excerpts:\n\n{excerpts}\n\nQuestion: {question}"


llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, rate_limiter=InMemoryRateLimiter(requests_per_second=CHAT_REQUESTS_PER_SECOND))
agent = build_agent()

### 2.3 Trying it out

The harder questions of section 1.4, plus two the rulebook cannot answer, each asked once since the agent has no memory yet. Under each answer, the pages of the chunks it was given.

In [14]:
# Constants
# Questions to try the agent: the harder ones of section 1.4, plus two the rulebook cannot answer
TRY_QUESTIONS = [question for question, _ in HARD_QUESTIONS] + [
    "¿Hay un tiempo máximo para cada turno?",      # not in the rulebook
    "¿Qué me recomiendas para cenar esta noche?",  # out of scope
]

In [15]:
def ask(question):
    """Question → agent → answer shown as Markdown, followed by the pages of the retrieved chunks."""
    result = agent.invoke({"messages": [HumanMessage(question)]})
    pages = ", ".join(f"p{chunk.metadata['page']}" for chunk in result["chunks"])
    display(Markdown(f"**{question}**\n\n{result['messages'][-1].text}\n\n*Retrieved: {pages}*\n\n---"))


for question in TRY_QUESTIONS:
    ask(question)

**¿Puedes pagar un dado con los recursos que obtienes de la propia acción?**

No, no puedes pagar un dado con los recursos que obtienes de la propia acción.

El procedimiento para usar los dados indica que primero debes colocar el dado (pagando o recibiendo la diferencia en monedas según su valor respecto a la casilla) y solo después realizas la acción de la carta o casilla. Por lo tanto, los recursos obtenidos al realizar la acción no están disponibles en el momento previo de pagar la diferencia del dado.

Lo tienes en la página 5 y 8.

*Retrieved: p8, p5, p5, p5, p17*

---

**¿Puedes ascender dos veces o colocar dos cortesanos en una misma acción?**

Sí, puedes hacer las dos acciones o incluso realizarlas una vez cada una.

Los cortesanos permiten hacer dos acciones distintas: solo una de ellas, o las dos acciones una vez cada una, pudiendo incluso usar fichas de cortesano distintas en el proceso.

Lo tienes en la página 10.

*Retrieved: p10, p10, p10, p10, p11*

---

**¿Puede un jardinero recién colocado activarse inmediatamente al final de una ronda?**

Sí.

Un jardinero recién colocado puede activarse al final de la ronda siempre que el puente que tiene encima conserve al menos un dado, ya que los jardineros se activan al terminar la ronda si hay dados en su puente y los jugadores resuelven las acciones de las cartas de jardín donde tengan jardineros de su clan.

Lo tienes en las páginas 10 y 13.

*Retrieved: p13, p10, p10, p17, p13*

---

**En The White Castle, un jugador quiere coger un dado de valor 2 y colocarlo en un espacio cuyo valor de referencia es 5. Tiene 1 moneda, por lo que no puede pagar las 3 monedas de diferencia. Sin embargo, al coger ese dado activa su bonificación de linterna, que le proporcionaría monedas suficientes para cubrir la diferencia. ¿Puede realizar la jugada? Explica el orden exacto de resolución, si puede utilizar la recompensa de la linterna para pagar el dado y qué alternativa tiene si no puede efectuar esa colocación.**

No, no puede realizar la jugada.

Al colocar el dado de valor 2 en un espacio con valor 5 (una diferencia de 3), el jugador debe pagar 3 monedas a la reserva común de inmediato en ese mismo paso de colocación. Dado que solo tiene 1 moneda, no es posible pagar la diferencia y, según las reglas, si no es posible pagar la diferencia, no se puede elegir ese dado para esa casilla. La bonificación de linterna se obtiene más tarde, después de haber elegido y colocado el dado de la parte izquierda del puente, por lo que esas monedas no están disponibles en el momento de comprobar si se puede pagar la diferencia de valor del dado.

Lo tienes en la página 5.

*Retrieved: p6, p1, p5, p7, p5*

---

**Al final de la primera ronda, un jugador activa un jardinero que ya tenía colocado. La acción de ese jardinero le permite colocar un segundo jardinero en otro jardín bajo un puente que contiene dados. El jugador sostiene que, como el segundo jardinero también está bajo un puente con dados, debe activar inmediatamente su acción de final de ronda. Además, afirma que no debe pagar de nuevo el coste de arroz de ese segundo jardinero, pero que tampoco necesita pagar las monedas exigidas por la acción que decida ejecutar. Evalúa por separado las tres afirmaciones. Después, explica qué sucede si la misma situación se produce al final de la tercera ronda.**

No, el jugador no tiene razón en ninguna de las tres afirmaciones.

Durante el final de la ronda, los jardineros se activan de forma general al cumplirse esa fase específica (segundo paso del final de ronda), no de manera inmediata al colocarse. Además, para colocar un nuevo jardinero se debe pagar su correspondiente coste de alimento (como indican las reglas de jardineros), y las reglas de los jardineros al final de ronda explican que se resuelven las acciones de las cartas de jardín donde se tengan jardineros, pero no eximen de pagar los costes adicionales que dichas acciones puedan exigir. Si esta misma situación se produjera al terminar la tercera ronda, tras reordenar el turno se pasaría directamente a la sección de Fin de partida sin resolver el paso de los jardines ni la preparación de la siguiente ronda.

Lo tienes en las páginas 10, 13 y 17.

*Retrieved: p13, p10, p13, p17, p13*

---

**¿Hay un tiempo máximo para cada turno?**

No encuentro esta regla en el manual, ya que el reglamento proporcionado no menciona si existe un tiempo máximo para cada turno. Es posible que el libro de reglas lo deje abierto o que la búsqueda no lo haya encontrado.

Como sugerencia de regla casera, podéis establecer un límite de 2 minutos por turno utilizando un reloj de arena o un temporizador para mantener el ritmo de la partida.

*Retrieved: p4, p13, p13, p12, p13*

---

**¿Qué me recomiendas para cenar esta noche?**

Solo puedo ayudarte con las reglas de White Castle.

*Retrieved: p20, p19, p2, p1, p13*

---

**What the answers show**
- **Format:** every answer follows its format. The question the rulebook does not cover gets "No encuentro esta regla en el manual", a labelled house rule and no page, and the one about dinner gets the fixed reply.
- **Paying for a die, short and long:** right, deduced from the order of the steps (see 1.4). The long one even reasons with the numbers of the situation.
- **Courtiers:** wrong. The search did its job (four chunks from page 10), but the model misreads "las dos acciones una vez cada una": each of the two actions can be done at most once. In earlier runs of the same prompt it got it right about half of the time: `gemini-3.5-flash-lite` ignores `temperature`, so its answers vary from one run to the next.
- **Gardeners:** the short question is ambiguous, and the answer is right for its literal reading (a gardener placed during the round is activated at the end of it). The long one is right about the immediate activation, the coins of the action and the third round. About the rice, it says the second gardener must pay it, which is what the rulebook says about placing gardeners; the claim in the question ("no pagar de nuevo") is itself ambiguous.
- **Where the errors come from:** in the wrong answer the right pages were retrieved, so the error comes from the reasoning of the generation step, not from the search. The chat model is the lightest free Gemini model; a larger one (`CHAT_MODEL`) could reason better, at the cost of daily quota.

## 3. Conversation memory

So far each question starts from zero: `generate` already passes the earlier messages to Gemini, but the graph forgets them as soon as it answers. Memory needs two pieces:
- **Keeping the conversation:** a **checkpointer** saves the state of the graph after every turn, under the id of its conversation (`thread_id`), and loads it back at the start of the next turn. Like a waiter's notepad with one sheet per table: each table only sees its own orders.
- **Searching with follow-up questions:** "¿Y si somos dos?" means nothing to the vector store on its own. A new node rewrites it first into a question that can be understood without the conversation.

```
START → rewrite → retrieve → generate → END
```

- **rewrite:** earlier turns + last question → kind of question and standalone question to search
- **retrieve:** the standalone question → the top `RETRIEVAL_K` chunks (as in section 2)
- **generate:** as in section 2, now with the whole saved conversation, plus a note when the player changes topic

### 3.1 Rewriting follow-up questions

From the second turn on, Gemini sorts the last question into one of two kinds:

| Kind | Example | Search with | Note to `generate` |
|---|---|---|---|
| `same_topic` | "¿Y luego cómo lo subo?" | the question rewritten to stand on its own (unchanged if it already does) | - |
| `new_topic` | a clear question about another topic | the question as it is | "the player has changed topic" |

- **Why the kind and not only the rewrite:** a first version only rewrote, and it tended to carry the earlier topic into clear questions (a question about stacking dice was searched "on the bridges" because the bridges had come up before). Naming the case of a new topic lets the rewrite leave it clean.
- **Why two kinds and not three:** a version with a third kind, clear questions about the same topic searched as written, sorted "¿Puedo hacer las dos cosas en el mismo turno?" there, so "las dos cosas" was searched as it was and the wrong rules came back. With two kinds, a doubtful question is at worst rewritten when it did not need it, which does no harm.
- **The rulebook's words:** the rewrite uses the words the agent used in its answers, which come from the rulebook, instead of the player's informal ones ("la sala del Daimio", not "arriba del todo"). The same goes for the steps of the "learning to play" mode: named by their content, not by their number ("the second step"), or the search misses them.
- **The note goes to `generate`, not to the search:** words from the earlier topic in the searched question would pull chunks of that topic. `generate` already sees the earlier turns, so the note only tells it not to mix them into the answer.
- **The rewrite is only for searching:** `generate` still answers the player's own words.
- **First turn:** there is nothing to rewrite, so no Gemini call is spent.

In [16]:
# Constants
REWRITE_PROMPT = """You prepare the last question of a conversation about the rules of the board game White Castle for a search in the rulebook. Classify it into one of two kinds and return the question to search, in Spanish:
- "same_topic": it continues the topic of the earlier turns. If it depends on them (for example "y si...", "en ese caso", "eso", "lo", "las dos cosas", "arriba del todo", "sí", "siguiente"), rewrite it so that it can be understood on its own, replacing the references with what they refer to. If it can already be understood on its own, return it unchanged. If the player only confirms or asks to go on, write a question about the step the assistant offered to explain next, named by its content, not by its number.
- "new_topic": it can be understood on its own and is about a different topic. Return it unchanged, without adding anything from the earlier turns.
When you rewrite, use the words the assistant used in the conversation (they come from the rulebook) instead of the player's informal ones: for example "la sala del Daimio" instead of "arriba del todo"."""

TOPIC_CHANGE_NOTE = "Note: the player has changed topic. The earlier turns are about other rules: answer this question on its own."

### 3.2 The graph with memory

The state of section 2 gains the kind of the last question and the question to search, and the graph is compiled with a checkpointer:
- **`InMemorySaver`:** keeps the conversations in memory, so they are lost when the kernel restarts. The demo and the review run in a single session, so saving them to disk would add a dependency for nothing.
- **Structured answer:** the rewrite asks Gemini for a small dictionary (kind and question) instead of free text, so the code can read the kind without parsing sentences.
- **`chat(question, thread_id)`:** sends one turn to the conversation `thread_id`. A new id starts a conversation from zero. Under each answer, the kind of question, the rewritten question when there is one, and the pages of the retrieved chunks, to tell a failed rewrite from a failed answer.

In [17]:
class MemoryState(AgentState):
    """
    State of the graph with memory:
        - The fields of AgentState (messages, chunks)
        - question_kind: first_turn, same_topic or new_topic
        - search_query: the last question, as it is searched in the rulebook
    """
    question_kind: str
    search_query: str


class Rewrite(TypedDict):
    """Structured answer of the rewrite: kind of the last question and the question to search."""
    kind: Literal["same_topic", "new_topic"]
    question: str


def build_memory_agent():
    """Graph START → rewrite → retrieve → generate → END, compiled with an in-memory checkpointer."""
    graph = StateGraph(MemoryState)
    graph.add_node("rewrite", rewrite)
    graph.add_node("retrieve", retrieve_search_query)
    graph.add_node("generate", generate_in_conversation)
    graph.add_edge(START, "rewrite")
    graph.add_edge("rewrite", "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)
    return graph.compile(checkpointer=InMemorySaver())


def rewrite(state):
    """Rewrite node: earlier turns + last question → kind + question to search (first turn: unchanged, no Gemini call)."""
    *history, question = state["messages"]
    if not history:
        return {"question_kind": "first_turn", "search_query": question.content}
    prompt = [SystemMessage(REWRITE_PROMPT), HumanMessage(build_rewrite_message(history, question.content))]
    result = rewriter.invoke(prompt)
    return {"question_kind": result["kind"], "search_query": result["question"]}


def build_rewrite_message(history, question):
    """Earlier turns as 'Jugador:' / 'Asistente:' lines + the last question."""
    speakers = {"human": "Jugador", "ai": "Asistente"}
    turns = "\n\n".join(f"{speakers[message.type]}: {message.text}" for message in history)
    return f"Conversation:\n\n{turns}\n\nLast question: {question}"


def retrieve_search_query(state):
    """Retrieve node: question to search → top RETRIEVAL_K chunks of the rulebook."""
    return {"chunks": vector_store.similarity_search(state["search_query"], k=RETRIEVAL_K)}


def generate_in_conversation(state):
    """Generate node with memory: as generate in section 2, with TOPIC_CHANGE_NOTE before a new-topic question."""
    if state["question_kind"] != "new_topic":
        return generate(state)
    *history, question = state["messages"]
    user_message = f"{TOPIC_CHANGE_NOTE}\n\n{build_user_message(question.content, state['chunks'])}"
    prompt = [SystemMessage(SYSTEM_PROMPT), *history, HumanMessage(user_message)]
    return {"messages": [llm.invoke(prompt)]}


def chat(question, thread_id):
    """
    One turn of the conversation thread_id → answer shown as Markdown:
        - Below it, how the question was searched and the pages of the retrieved chunks
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = memory_agent.invoke({"messages": [HumanMessage(question)]}, config)
    pages = ", ".join(f"p{chunk.metadata['page']}" for chunk in result["chunks"])
    display(Markdown(f"**{question}**\n\n{result['messages'][-1].text}\n\n*{describe_search(question, result)} · Retrieved: {pages}*\n\n---"))


def describe_search(question, result):
    """
    How the question was searched: its kind, plus the rewritten question if it changed
        Example: 'same_topic → ¿Cómo sube un cortesano de piso?'
    """
    if result["search_query"] != question:
        return f"{result['question_kind']} → {result['search_query']}"
    return result["question_kind"]


rewriter = llm.with_structured_output(Rewrite)
memory_agent = build_memory_agent()

### 3.3 Trying it out

Five conversations, each in its own thread. The first three have five turns each, and each one stresses a different kind of question:
- **dice:** clear questions about one topic (using the dice), so nothing should need rewriting. One of them calls the lantern of the bridge "farol", a word the rulebook does not use.
- **courtiers:** follow-ups about one topic (courtiers), which only make sense with the earlier turns.
- **topic_changes:** follow-ups mixed with changes of topic, without announcing them: resources, then gardeners, then breaking a tie.
- **new_thread:** a follow-up from `courtiers` in a new conversation, which must not remember it.
- **tutorial:** the "learning to play" mode of the system prompt, which explains one step at a time and needs the player to confirm.

The answers are checked against the rulebook in the conclusions below.

In [18]:
# Constants
# Conversations to try the memory: thread_id → questions, in order
MEMORY_CONVERSATIONS = {
    "dice": [
        "¿De qué parte del puente puedo coger un dado?",
        "¿Qué pasa si pongo un dado más alto que el número de la casilla?",
        "¿Qué pasa si no puedo pagar la diferencia al poner un dado más bajo que la casilla?",
        "¿Qué gano si cojo el dado que está junto al farol del puente?",   # "farol": the rulebook says "linterna"
        "¿Cuántos dados se pueden poner en el pozo?",
    ],
    "courtiers": [
        "¿Cómo meto un cortesano en el castillo?",
        "¿Y luego cómo lo subo?",                                          # "lo": the courtier
        "¿Puedo hacer las dos cosas en el mismo turno?",                   # entering and going up
        "¿Y qué gano si llega arriba del todo?",                           # the Daimio's hall
        "¿Cuántos puntos me da al final de la partida?",                   # a courtier on the third floor
    ],
    "topic_changes": [
        "¿Cuánto hierro puedo tener como máximo?",
        "¿Y si gano más y me paso?",                                       # follow-up
        "¿Cuándo se activan los jardineros?",                              # new topic, not announced
        "¿Y en la última ronda también?",                                  # follow-up about gardeners
        "Vale. ¿Cómo se desempata al final?",                              # new topic
    ],
    "new_thread": [
        "¿Y luego cómo lo subo?",                                          # follow-up with no conversation behind it
    ],
    "tutorial": [
        "Enséñame a jugar",
        "Sí, empecemos",                                                   # confirmation → first step
        "Vale, siguiente paso",                                            # confirmation → second step
    ],
}

In [19]:
for thread_id, questions in MEMORY_CONVERSATIONS.items():
    display(Markdown(f"#### Conversation `{thread_id}`"))
    for question in questions:
        chat(question, thread_id)

#### Conversation `dice`

**¿De qué parte del puente puedo coger un dado?**

Sí, solo puedes tomar un dado de cualquiera de los 2 extremos del puente que prefieras.

Los dados de la parte central del puente no están disponibles para ser cogidos directamente. Además, cuando tomas un dado de un extremo, si quedan dados en la parte central, debes desplazar el más próximo al hueco que acabas de dejar para que quede disponible para el siguiente jugador.

Lo tienes en la página 4.

*first_turn · Retrieved: p4, p2, p5, p4, p4*

---

**¿Qué pasa si pongo un dado más alto que el número de la casilla?**

Obtendrás la diferencia en monedas de la reserva común.

Cuando colocas un dado en una casilla del tablero central o del dominio personal sobre un valor impreso o el de otro dado, si el valor del dado que colocas es superior, la regla indica que se gana esa diferencia exacta en monedas sacándolas de la reserva común.

Lo tienes en la página 5.

*new_topic · Retrieved: p5, p5, p17, p4, p2*

---

**¿Qué pasa si no puedo pagar la diferencia al poner un dado más bajo que la casilla?**

No se puede elegir ese dado.

Si el valor del dado que colocas es inferior al de la casilla, debes pagar la diferencia en monedas a la reserva común. Si no tienes suficientes monedas para realizar ese pago, la regla indica explícitamente que no está permitido elegir ese dado.

Lo tienes en la página 5.

*new_topic · Retrieved: p5, p17, p5, p4, p4*

---

**¿Qué gano si cojo el dado que está junto al farol del puente?**

Obtendrás la recompensa de lámpara de tu dominio personal.

Cuando eliges un dado de la parte izquierda del puente, que es la zona situada junto a la linterna o farol, la regla establece específicamente que recibes dicha recompensa.

Lo tienes en la página 5.

*new_topic · Retrieved: p4, p5, p2, p4, p4*

---

...

**¿Y luego cómo lo subo?**

No encuentro esta regla en el manual, ya que el texto no especifica qué hacer para subir el marcador de recursos más allá de la regla general de que subes el cubo cuando los obtienes. Puede que el manual no lo detalle o que la búsqueda no lo haya encontrado.

Como sugerencia de regla casera, podéis acordar que al realizar ciertas acciones secundarias o bonificaciones del tablero podáis avanzar el marcador del recurso correspondiente en una unidad.

*first_turn · Retrieved: p13, p16, p5, p4, p2*

---

#### Conversation `tutorial`

**Enséñame a jugar**

1. Aquí tienes los primeros pasos para aprender a jugar a The White Castle:
   1. Preparación del castillo (página 2).
   2. Preparación de los jugadores (página 3).
   3. Dinámica de la partida y rondas (página 4).
2. ¿Estás listo para el primer paso?
3. (Esperando tu confirmación para explicar el primer paso).

*first_turn · Retrieved: p3, p2, p3, p4, p2*

---

**Sí, empecemos**

1. Vamos con el primer paso: Preparación del castillo.
2. Colocad el tablero central, barajad y colocad los mazos de mayordomos y diplomáticos con una carta visible en cada espacio, poned la carta de favor del Daimio en el último piso, colocad las losetas de dado al azar en las salas y en el pozo asegurando que haya al menos 2 colores diferentes por sala, y situad la ficha de ronda en el primer espacio de su marcador (dejando la reserva común de dinero, sellos del Daimio y fichas de +40/+80 a un lado). Lo tienes en la página 2.
3. ¿Estás listo para el siguiente paso?

*same_topic → Preparación del castillo · Retrieved: p2, p2, p2, p6, p6*

---

**Vale, siguiente paso**

1. Vamos con el siguiente paso: Preparación de los jugadores.
2. Cada jugador toma un tablero personal y coloca sus 15 miembros del clan (5 cortesanos, 5 jardineros y 5 guerreros) en sus espacios correspondientes, sitúa sus contadores de recursos en el espacio 0, coloca la ficha de abanico, organiza el orden de turno y el orden de influencia con las fichas de garza y grulla, y finalmente elige una pareja de cartas iniciales (recursos, acción y opcionalmente decreto) en orden inverso de turno. Lo tienes en la página 3.
3. ¿Estás listo para el siguiente paso?

*same_topic → Preparación de los jugadores · Retrieved: p3, p3, p3, p2, p2*

---

**What the conversations show**

Checked against the rulebook, 13 of the 15 answers of the three long conversations are right, and the memory holds for the five turns of each one. In neither error does the agent forget the conversation: one is a rewrite that loses a word, the other a rule split across two chunks.

- **Memory holds over five turns:** every follow-up is understood ("lo" is the courtier, "las dos cosas" are entering the castle and going up, "me paso" is the iron, "la última ronda" is about gardeners), and no answer mixes in an earlier topic, even after two changes of topic in `topic_changes`.
- **The rewrite makes follow-ups searchable:** "¿Puedo hacer las dos cosas en el mismo turno?" is searched as "¿Puedo solicitar audiencia y ascender un cortesano en el mismo turno?", and the courtier rule of page 10 comes first. In the version with three kinds it was searched as written and missed it.
- **The rulebook's words find more than the player's:** "¿Cuántos puntos me da al final de la partida?" is searched with "la sala del Daimio", and the scoring table of page 14 is retrieved. With "arriba del todo", the player's words, an earlier run missed it.
- **Error 1, a rewrite that drops the subject:** "¿Y qué gano si llega arriba del todo?" was rewritten as "¿Qué gano si llega a la sala del Daimio?", without the courtier, so the search brought pages 12, 6, 8 and 16 instead of page 11, and the agent said the rule was not in the rulebook. Each run of this conversation got one different follow-up wrong: `gemini-3.5-flash-lite` does not rewrite the same way every time.
- **Error 2, not a memory error:** "¿Y en la última ronda también?" is rewritten well and the gardeners chunk of page 13 comes first, but the exception (after the third round the game ends without activating the gardeners) is in the chunk before it, under "Nuevo orden de turno", which is not retrieved. The agent answers "sí", which is wrong.
- **Same or new topic is not reliable, but harmless:** in `dice`, clear questions about the same topic are often classified `new_topic`. Both kinds are searched as written, and the note only asks `generate` to answer the question on its own.
- **Vocabulary:** "farol" finds the lantern rule (page 5): the embeddings match it by meaning.
- **A follow-up with no conversation** (`new_thread`) is not understood, as expected, but the agent guesses a meaning (resource markers) instead of asking what the player means.
- **Learning to play:** it keeps track of the steps over three turns and each confirmation finds the page of its step. The steps still come as a numbered list of the three parts of the answer format.

## 4. Chat and examples

*To be built.* A chat cell to talk to the agent, and at least five example questions with their answers.